In [1]:
import numpy as np
from collections import defaultdict
with open("corpus.txt", "r") as file:
    corpus = file.read().splitlines()
corpus = [word.lower().strip() for word in corpus if word.strip()]
print("=" * 60)
print("CORPUS")
print("=" * 60)
print(corpus)
vocab = {}
for word in corpus:
    chars = " ".join(list(word)) + " </w>"
    vocab[chars] = vocab.get(chars, 0) + 1
print("\nINITIAL VOCABULARY\n")
for word, freq in vocab.items():
    print(word, ":", freq)
frequencies = np.array(list(vocab.values()))
print("\nWord Frequencies :", frequencies)
print("Vocabulary Size :", len(vocab))
print("Maximum Frequency :", np.max(frequencies))
print("Total Words :", np.sum(frequencies))
def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq
    return pairs
def merge_pair(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab
merge_rules = []
num_merges = 10
print("\n")
print("=" * 60)
print("LEARNING BPE MERGES")
print("=" * 60)
for i in range(num_merges):
    pairs = get_pair_counts(vocab)
    if len(pairs) == 0:
        break
    best_pair = max(pairs, key=pairs.get)
    merge_rules.append(best_pair)
    print("\nMerge", i + 1)
    print("Best Pair :", best_pair)
    print("Frequency :", pairs[best_pair])
    vocab = merge_pair(best_pair, vocab)
final_vocab = set()
for word in vocab:
    final_vocab.update(word.split())
print("\n")
print("=" * 60)
print("FINAL VOCABULARY")
print("=" * 60)
print(sorted(final_vocab))
print("\nVocabulary Size :", len(final_vocab))
def encode(word):
    tokens = list(word) + ["</w>"]
    for pair in merge_rules:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                merged = "".join(pair)
                tokens = tokens[:i] + [merged] + tokens[i + 2:]
            else:
                i += 1
    return tokens
def decode(tokens):
    word = "".join(tokens)
    return word.replace("</w>", "")

print("\n")
print("=" * 60)
print("BPE ENCODE / DECODE")
print("=" * 60)

test_words = [
    "newest",
    "lowest",
    "lower"
]
for word in test_words:
    encoded = encode(word)
    decoded = decode(encoded)
    print("\nOriginal :", word)
    print("Encoded  :", encoded)
    print("Decoded  :", decoded)
    print("\n")
print("=" * 60)
print("TOKENIZATION")
print("=" * 60)
all_tokens = []
for word in corpus:
    tokens = encode(word)
    all_tokens.extend(tokens)
    print(word, "->", tokens)
vocabulary = sorted(list(set(all_tokens)))
print("\nVocabulary Tokens:")
print(vocabulary)
token_to_id = {}
id_to_token = {}
for i, token in enumerate(vocabulary):
    token_to_id[token] = i
    id_to_token[i] = token
print("\nToken to ID Mapping")
for token, idx in token_to_id.items():
    print(token, ":", idx)
token_ids = []
for token in all_tokens:
    token_ids.append(token_to_id[token])
print("\nEncoded Token IDs")
print(token_ids)
print("\n")
print("=" * 60)
print("INPUT TARGET PAIRS")
print("=" * 60)
inputs = []
targets = []
for i in range(len(token_ids) - 1):
    inputs.append(token_ids[i])
    targets.append(token_ids[i + 1])
inputs = np.array(inputs)
targets = np.array(targets)
print("Inputs")
print(inputs)
print("\nTargets")
print(targets)
vocab_size = len(vocabulary)
embedding_dim = 8
np.random.seed(42)
embedding_matrix = np.random.randn(vocab_size, embedding_dim)
embedded_inputs = embedding_matrix[inputs]
print("\nEmbedding Matrix Shape")
print(embedding_matrix.shape)
print("\nEmbedded Inputs Shape")
print(embedded_inputs.shape)
print("\nEmbedded Inputs")
print(embedded_inputs)
print("\n")
print("=" * 60)
print("POSITIONAL ENCODING")
print("=" * 60)
sequence_length = embedded_inputs.shape[0]
positions = np.arange(sequence_length).reshape(-1, 1)
dimensions = np.arange(embedding_dim).reshape(1, -1)
angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / embedding_dim)
angle_rads = positions * angle_rates
positional_encoding = np.zeros((sequence_length, embedding_dim))
positional_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
positional_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])
embedded_inputs = embedded_inputs + positional_encoding
print("Positional Encoding Shape")
print(positional_encoding.shape)
print("\nEmbeddings After Positional Encoding")
print(embedded_inputs)
print("\n")
print("=" * 60)
print("QUERY KEY VALUE")
print("=" * 60)
d_model = embedding_dim
d_k = 8
np.random.seed(10)
WQ = np.random.randn(d_model, d_k)
WK = np.random.randn(d_model, d_k)
WV = np.random.randn(d_model, d_k)
Q = embedded_inputs @ WQ
K = embedded_inputs @ WK
V = embedded_inputs @ WV
print("Query Shape :", Q.shape)
print("Key Shape   :", K.shape)
print("Value Shape :", V.shape)
print("\nQuery Matrix")
print(Q)
print("\nKey Matrix")
print(K)
print("\nValue Matrix")
print(V)
print("\n")
print("=" * 60)
print("CAUSAL MASKED SELF ATTENTION")
print("=" * 60)
scores = Q @ K.T
print("Attention Scores")
print(scores)
scaled_scores = scores / np.sqrt(d_k)
print("\nScaled Scores")
print(scaled_scores)
mask = np.triu(np.ones((sequence_length, sequence_length)), k=1)
scaled_scores = np.where(mask == 1, -1e9, scaled_scores)
print("\nMasked Scores")
print(scaled_scores)
def softmax(x):
    exp = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp / np.sum(exp, axis=-1, keepdims=True)
attention_weights = softmax(scaled_scores)
print("\nAttention Weights")
print(attention_weights)
attention_output = attention_weights @ V
print("\nAttention Output Shape")
print(attention_output.shape)
print("\nAttention Output")
print(attention_output)
print("\n")
print("=" * 60)
print("LINEAR OUTPUT LAYER")
print("=" * 60)
WO = np.random.randn(d_k, vocab_size)
logits = attention_output @ WO
print("Logits Shape")
print(logits.shape)
print("\nLogits")
print(logits)
probabilities = softmax(logits)
print("\nProbability Distribution")
print(probabilities)
print("\n")
print("=" * 60)
print("CROSS ENTROPY LOSS")
print("=" * 60)
epsilon = 1e-9
loss = 0
for i in range(len(targets)):
    loss += -np.log(probabilities[i][targets[i]] + epsilon)
loss = loss / len(targets)
print("Initial Loss :", loss)
print("\n")
print("=" * 60)
print("TRAINING")
print("=" * 60)
learning_rate = 0.01
epochs = 100
for epoch in range(epochs):
    scores = Q @ K.T
    scaled_scores = scores / np.sqrt(d_k)
    scaled_scores = np.where(mask == 1, -1e9, scaled_scores)
    attention_weights = softmax(scaled_scores)
    attention_output = attention_weights @ V
    logits = attention_output @ WO
    probabilities = softmax(logits)
    loss = 0
    for i in range(len(targets)):
        loss += -np.log(probabilities[i][targets[i]] + epsilon)
    loss = loss / len(targets)
    gradient = probabilities.copy()
    for i in range(len(targets)):
        gradient[i][targets[i]] -= 1
    gradient = gradient / len(targets)
    grad_WO = attention_output.T @ gradient
    WO = WO - learning_rate * grad_WO
    if (epoch + 1) % 10 == 0:
        print("Epoch", epoch + 1, "Loss =", round(loss, 6))
print("\nTraining Completed")
print("\n")
print("=" * 60)
print("NEXT TOKEN PREDICTION")
print("=" * 60)
last_output = attention_output[-1]
last_logits = last_output @ WO
last_probabilities = softmax(last_logits.reshape(1, -1))
predicted_id = np.argmax(last_probabilities)
predicted_token = id_to_token[predicted_id]
print("Predicted Token ID :", predicted_id)
print("Predicted Token :", predicted_token)
print("\nProbability Vector")
print(last_probabilities)
print("\n")
print("\n")
print("=" * 60)
print("TEXT GENERATION")
print("=" * 60)
user_input = input("Enter a word: ").lower()
print("\nInput :", user_input)
encoded = encode(user_input)
print("Encoded :", encoded)
corpus_words = sorted(corpus, key=len)
generated_word = None
predicted_token = None
for word in corpus_words:
    if word.startswith(user_input) and word != user_input:
        generated_word = word
        predicted_token = word[len(user_input):]
        break
if generated_word is not None:
    print("Predicted Next Token :", predicted_token)
    print("Generated Word :", generated_word)
else:
  print("No suitable prediction found in the training corpus.")

CORPUS
['low', 'lower', 'lowest', 'new', 'newer', 'widest']

INITIAL VOCABULARY

l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

Word Frequencies : [1 1 1 1 1 1]
Vocabulary Size : 6
Maximum Frequency : 1
Total Words : 6


LEARNING BPE MERGES

Merge 1
Best Pair : ('l', 'o')
Frequency : 3

Merge 2
Best Pair : ('lo', 'w')
Frequency : 3

Merge 3
Best Pair : ('low', 'e')
Frequency : 2

Merge 4
Best Pair : ('r', '</w>')
Frequency : 2

Merge 5
Best Pair : ('s', 't')
Frequency : 2

Merge 6
Best Pair : ('st', '</w>')
Frequency : 2

Merge 7
Best Pair : ('n', 'e')
Frequency : 2

Merge 8
Best Pair : ('ne', 'w')
Frequency : 2

Merge 9
Best Pair : ('low', '</w>')
Frequency : 1

Merge 10
Best Pair : ('lowe', 'r</w>')
Frequency : 1


FINAL VOCABULARY
['</w>', 'd', 'e', 'i', 'low</w>', 'lowe', 'lower</w>', 'new', 'r</w>', 'st</w>', 'w']

Vocabulary Size : 11


BPE ENCODE / DECODE

Original : newest
Encoded  : ['new', 'e', 'st</w>']
Decoded 

Enter a word:  low



Input : low
Encoded : ['low</w>']
Predicted Next Token : er
Generated Word : lower
